# Vision OCR + Upstage 검증 구조 시험

동일 PDF 페이지의 Vision OCR과 Upstage Document Parse 원문을 비교한 뒤, 원본 PDF 직접 검수 필드로 차이를 판정합니다. 기존 정답 데이터셋과 기존 03 평가 결과는 수정하지 않습니다.

In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import html
import json
import re

import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / 'data').is_dir() and (cwd / 'notebooks').is_dir():
    BACKEND_ROOT = cwd
elif (cwd.parent / 'data').is_dir() and cwd.name == 'notebooks':
    BACKEND_ROOT = cwd.parent
else:
    raise RuntimeError(f'프로젝트 루트 또는 notebooks 폴더를 찾을 수 없습니다: {cwd}')

PROJECT_ROOT = BACKEND_ROOT
VISION_DIR = BACKEND_ROOT / 'data' / 'documents' / 'vision'
OCR_ROOT = BACKEND_ROOT / 'notebooks' / 'data' / '03_ocr_engine_comparison'
FIELD_GOLD_PATH = BACKEND_ROOT / 'notebooks' / 'data' / '03_ocr_engine_comparison' / '03_manual_field_gold.json'
RESULT_PATH = OCR_ROOT / 'vision_upstage_validation_results.json'
REPORT_PATH = PROJECT_ROOT / 'docs' / '02_vision_paddle_upstage_ocr_comparison_report.html'

# 기본값은 캐시된 원문 사용. 신규 PDF에서는 Upstage 호출 결과를 upstage_raw에 저장한 후 이 비교 셀을 실행합니다.
USE_CACHED_UPSTAGE = True


In [2]:
def sample_key(case: dict) -> str:
    return f"{case['issuer']}__{Path(case['file_name']).stem}__p{case['page_number']:03d}"

def normalize(text: str) -> str:
    text = html.unescape(text or '')
    text = re.sub(r'<br\s*/?>', ' ', text, flags=re.I)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^0-9A-Za-z가-힣%.,:/+~()\-]+', ' ', text.lower())
    return re.sub(r'\s+', ' ', text).strip()

def read_vision_page(case: dict) -> str:
    stem = Path(case['file_name']).stem
    matches = list(VISION_DIR.rglob(stem + '.txt'))
    if len(matches) != 1:
        raise RuntimeError(f'Vision 원문을 하나로 찾지 못했습니다: {stem}: {matches}')
    document = matches[0].read_text(encoding='utf-8')
    markers = list(re.finditer(r'^\[PAGE (\d+)\]\s*$', document, re.M))
    for index, marker in enumerate(markers):
        if int(marker.group(1)) == case['page_number']:
            end = markers[index + 1].start() if index + 1 < len(markers) else len(document)
            return document[marker.end():end]
    raise RuntimeError(f"Vision page 누락: {case['file_name']} p{case['page_number']}")

def read_upstage_page(case: dict) -> str:
    path = OCR_ROOT / 'upstage_raw' / f"{sample_key(case)}.json"
    if not path.exists():
        raise RuntimeError(f'Upstage 캐시 원문이 없습니다: {path}')
    payload = json.loads(path.read_text(encoding='utf-8'))
    # 원문 응답은 PDF 전체 페이지를 포함하므로 지정 page_number만 선택한다.
    parts = []
    for element in payload.get('elements', []):
        if element.get('page') != case['page_number']:
            continue
        content = element.get('content', {})
        parts.append(content.get('text') or content.get('markdown') or content.get('html') or '')
    return '\n'.join(parts)

def token_recall(reference: str, candidate: str) -> float:
    ref, got = Counter(reference.split()), Counter(candidate.split())
    return sum((ref & got).values()) / max(sum(ref.values()), 1)

def numeric_tokens(text: str) -> set[str]:
    return set(re.findall(r'(?<![0-9])[0-9][0-9,./%~:-]*(?![0-9])', text))


In [3]:
field_cases = json.loads(FIELD_GOLD_PATH.read_text(encoding='utf-8'))['cases']
comparison_rows, adjudication_rows = [], []

for case in field_cases:
    vision = normalize(read_vision_page(case))
    upstage = normalize(read_upstage_page(case))
    vision_numbers, upstage_numbers = numeric_tokens(vision), numeric_tokens(upstage)
    comparison_rows.append({
        'issuer': case['issuer'],
        'file_name': case['file_name'],
        'page_number': case['page_number'],
        'vision_chars': len(vision),
        'upstage_chars': len(upstage),
        'vision_to_upstage_token_recall': token_recall(upstage, vision),
        'upstage_to_vision_token_recall': token_recall(vision, upstage),
        'vision_only_numeric': sorted(vision_numbers - upstage_numbers),
        'upstage_only_numeric': sorted(upstage_numbers - vision_numbers),
    })

    # 2차 판정: 원본 PDF를 직접 보고 만든 필드값을 독립 기준으로 사용한다.
    for field_name, raw_value in case['fields'].items():
        value = normalize(raw_value)
        vision_ok, upstage_ok = value in vision, value in upstage
        if vision_ok and upstage_ok:
            verdict = 'both_correct_for_manual_field'
        elif vision_ok:
            verdict = 'upstage_error_or_omission'
        elif upstage_ok:
            verdict = 'vision_error_or_omission'
        else:
            verdict = 'both_error_or_omission'
        adjudication_rows.append({
            'issuer': case['issuer'], 'file_name': case['file_name'],
            'page_number': case['page_number'], 'field_name': field_name,
            'source_verified_value': raw_value, 'verdict': verdict,
        })

comparison = pd.DataFrame(comparison_rows)
adjudication = pd.DataFrame(adjudication_rows)
display(comparison[['issuer', 'file_name', 'page_number', 'vision_chars', 'upstage_chars', 'vision_to_upstage_token_recall', 'upstage_to_vision_token_recall', 'vision_only_numeric', 'upstage_only_numeric']])
display(adjudication['verdict'].value_counts().rename_axis('verdict').reset_index(name='field_count'))


,issuer,file_name,page_number,vision_chars,upstage_chars,vision_to_upstage_token_recall,upstage_to_vision_token_recall,vision_only_numeric,upstage_only_numeric
0,BC,BC_Baro_Clear_Plus.pdf,2,2571,2588,0.953448,0.971880,"[1588-4466, 1899-7771]","[588-4466, 899-7771]"
1,NH,NH_AllWonderful.pdf,3,5927,5863,0.907173,0.907939,[2026.02.10.],[2026.02.]
2,hana,Hana_Everyones_Shinsegae.pdf,1,4487,4517,0.858775,0.866876,"[2025-, 2025.06.25]","[2,2, 2021-, 2021.06.2]"
3,hyundai,Hyundai_T_20260319.pdf,5,689,611,0.974684,0.962500,[],"[02, 03]"
4,ibk,IBK_DailyWith.pdf,2,685,574,0.937008,0.908397,[],[]
5,kookmin,Kookmin_My_WE_SH_20250102.pdf,5,911,832,0.981481,0.981481,[],[]
6,lotte,Lotte_LOCA_LIKIT_Play.pdf,4,929,846,0.945355,0.925134,[],[]
7,samsung,Samsung_5_V4.pdf,2,1828,1752,0.972973,0.980926,[],[]
8,shinhan,Shinhan_Discount_Plan+_20250509.pdf,3,1786,1677,0.997333,0.984211,[],[]
9,woori,Woori_7CORE.pdf,2,3099,3006,0.821898,0.818314,[],[]


,verdict,field_count
0,both_correct_for_manual_field,43


In [4]:
result = {
    'scope': {'pages': len(comparison), 'manual_source_verified_fields': len(adjudication), 'upstage_source': 'cached raw response'},
    'comparison': comparison.to_dict(orient='records'),
    'adjudication': adjudication.to_dict(orient='records'),
    'adjudication_summary': adjudication['verdict'].value_counts().to_dict(),
}
RESULT_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')

summary_rows = ''.join(f'<tr><td>{html.escape(name)}</td><td>{count}</td></tr>' for name, count in result['adjudication_summary'].items())
page_rows = ''.join(
    f"<tr><td>{html.escape(row['issuer'])}</td><td>{html.escape(row['file_name'])}</td><td>{row['page_number']}</td><td>{row['vision_chars']}</td><td>{row['upstage_chars']}</td><td>{row['vision_to_upstage_token_recall'] * 100:.2f}%</td><td>{row['upstage_to_vision_token_recall'] * 100:.2f}%</td><td>{html.escape(', '.join(row['vision_only_numeric']) or '-')}</td><td>{html.escape(', '.join(row['upstage_only_numeric']) or '-')}</td></tr>"
    for row in result['comparison']
)
field_rows = ''.join(
    f"<tr><td>{html.escape(row['issuer'])}</td><td>{html.escape(row['file_name'])}</td><td>{row['page_number']}</td><td>{html.escape(row['field_name'])}</td><td>{html.escape(row['source_verified_value'])}</td><td>{'두 엔진 모두 정상' if row['verdict'] == 'both_correct_for_manual_field' else html.escape(row['verdict'])}</td></tr>"
    for row in result['adjudication']
)
section = f'''<section id="vision-upstage-validation"><h2>Vision OCR + Upstage 검증 구조 시험</h2><p><b>1차 비교:</b> 동일한 10개 PDF 지정 페이지의 Vision OCR과 Upstage 원문을 공통 정규화 뒤 비교했습니다. <b>2차 원본 대조:</b> 원본 PDF를 직접 확인해 만든 43개 필드값을 독립 기준으로 양쪽 출력에 대조했습니다. 기존 전체 페이지 정답셋과 기존 성능 결과는 변경하지 않았습니다.</p><h3>2차 필드 기준 판정 결과</h3><table><thead><tr><th>결과</th><th>필드 수</th><th>의미</th></tr></thead><tbody><tr><td>두 엔진 모두 정상</td><td>{result['adjudication_summary'].get('both_correct_for_manual_field', 0)}</td><td>원본 검수 필드값이 Vision과 Upstage 양쪽에 모두 존재</td></tr><tr><td>Vision 오류 또는 누락</td><td>{result['adjudication_summary'].get('vision_error_or_omission', 0)}</td><td>Upstage에만 원본 검수 필드값 존재</td></tr><tr><td>Upstage 오류 또는 누락</td><td>{result['adjudication_summary'].get('upstage_error_or_omission', 0)}</td><td>Vision에만 원본 검수 필드값 존재</td></tr><tr><td>두 엔진 모두 오류 또는 누락</td><td>{result['adjudication_summary'].get('both_error_or_omission', 0)}</td><td>양쪽 모두 원본 검수 필드값 미포함</td></tr></tbody></table><p><b>이번 시험 결과:</b> 43개 필드 모두 ‘두 엔진 모두 정상’으로 판정됐습니다. 이는 필드 수준 결과이며, 전체 본문이 완전히 동일하거나 오류가 없다는 뜻은 아닙니다.</p><h4>2차 필드별 판정 상세</h4><table><thead><tr><th>카드사</th><th>PDF</th><th>쪽</th><th>필드명</th><th>원본 검수값</th><th>판정</th></tr></thead><tbody>{field_rows}</tbody></table><details><summary>기술 판정 원문 보기</summary><table><thead><tr><th>판정 코드</th><th>필드 수</th></tr></thead><tbody>{summary_rows}</tbody></table></details><h3>1차 페이지별 비교</h3><table><thead><tr><th>카드사</th><th>PDF</th><th>쪽</th><th>Vision 문자 수</th><th>Upstage 문자 수</th><th>Vision→Upstage 토큰 보존</th><th>Upstage→Vision 토큰 보존</th><th>Vision 단독 숫자</th><th>Upstage 단독 숫자</th></tr></thead><tbody>{page_rows}</tbody></table><h3>운영 규칙</h3><ol><li>Vision과 Upstage의 필드값이 모두 일치하면 자동 통과 후보로 둡니다.</li><li>한쪽만 값이 있거나 값이 다르면 원본 PDF 대조 대기열로 보냅니다.</li><li>원본 대조 뒤 Vision 오류, Upstage 오류, 양쪽 오류, 정규화/필드 추출 규칙 문제로 기록합니다.</li><li>현재 PaddleOCR은 별도 시험에서 핵심 필드 누락이 확인되어 이 교차 검증 경로에는 포함하지 않습니다.</li></ol><p><b>한계:</b> 이번 2차 판정은 43개 수동 필드에만 근거합니다. 본문 전체의 중복·문장 순서·비필드 문구 오류는 전체 원문 전사가 보완된 뒤 별도로 판정해야 합니다.</p></section>'''
# 표시 순서도 실제 검증 흐름과 같게 1차 비교 후 2차 원본 판정으로 정렬한다.
intro, after_second = section.split('<h3>2차 필드 기준 판정 결과</h3>', 1)
second_block, after_first = after_second.split('<h3>1차 페이지별 비교</h3>', 1)
first_block, tail = after_first.split('<h3>운영 규칙</h3>', 1)
first_phase_note = '''<p><b>1차 비교 컬럼 설명:</b></p><ul><li><b>Vision/Upstage 문자 수</b>: 각 엔진이 지정 페이지에서 추출한 정규화 텍스트 길이입니다. 큰 차이는 내용 누락, 중복, 페이지 요소 처리 차이의 후보 신호입니다.</li><li><b>Vision→Upstage 토큰 보존</b>: Vision에 있는 단어 중 Upstage에도 있는 비율입니다.</li><li><b>Upstage→Vision 토큰 보존</b>: Upstage에 있는 단어 중 Vision에도 있는 비율입니다.</li><li><b>단독 숫자</b>: 한 엔진에만 등장한 숫자 토큰입니다. 한도·할인율·연회비·연락처 등의 값 차이 후보이므로, 발견되면 2차 원본 대조 대상으로 봅니다.</li></ul><p>1차 비교는 차이 후보를 찾는 단계이며, 어느 엔진이 오류인지는 이 표만으로 판정하지 않습니다.</p>'''
section = intro + '<h3>1차 페이지별 비교</h3>' + first_block + first_phase_note + '<h3>2차 필드 기준 판정 결과</h3>' + second_block + '<h3>운영 규칙</h3>' + tail
report = REPORT_PATH.read_text(encoding='utf-8')
report = re.sub(r'<section id="vision-upstage-validation">.*?</section>', '', report, flags=re.S)
REPORT_PATH.write_text(report.replace('</body>', section + '</body>'), encoding='utf-8')
print(f'검증 결과: {RESULT_PATH}')
print(f'보고서 반영: {REPORT_PATH}')
print(result['adjudication_summary'])


검증 결과: notebooks/data/03_ocr_engine_comparison/vision_upstage_validation_results.json
보고서 반영: /home/sms/openclaw_file/RAIchU/docs/02_vision_paddle_upstage_ocr_comparison_report.html
{'both_correct_for_manual_field': 43}
